In [3]:
!pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Trupal Dholariya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   --------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Trupal Dholariya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import os
import warnings
import logging
from pathlib import Path

# ── Third-Party ───────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # non-interactive backend (safe for servers)
import matplotlib.pyplot  as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import joblib

from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier

# ── Global Config ─────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
log = logging.getLogger("SolarML")

from pathlib import Path

BASE_DIR = Path.cwd()

DATASET_PATH = BASE_DIR / "solar_ml_master_dataset.xlsx"
MODELS_DIR = BASE_DIR / "models"
PLOTS_DIR = BASE_DIR / "plots"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE  = 42
TARGET_COL    = "future_failure_7_10_days"
N_SPLITS_CV   = 5
FAILURE_PROB_THRESHOLD = 0.70


# =============================================================================
# STEP 1 – LOAD & INSPECT DATASET
# =============================================================================

def load_dataset(path: Path) -> pd.DataFrame:
    """Load the master dataset from an Excel or CSV file.

    The function auto-detects the file extension so it works with both
    .xlsx  (the current file in the workspace) and .csv variants.
    """
    log.info("Loading dataset from: %s", path)

    ext = path.suffix.lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(path, engine="openpyxl")
    elif ext == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    log.info("Raw shape: %s", df.shape)
    log.info("Columns  : %s", df.columns.tolist())
    log.info("Dtypes\n%s", df.dtypes)
    log.info("First 5 rows\n%s", df.head())

    return df


def preprocess_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Detect and convert the datetime column; sort by inverter_id + datetime."""

    # ── Detect datetime column ─────────────────────────────────────────────
    dt_candidates = [c for c in df.columns
                     if any(k in c.lower() for k in ("datetime", "timestamp",
                                                       "date", "time"))]
    if not dt_candidates:
        raise ValueError("No datetime-like column found. "
                         "Please rename your time column to contain 'datetime'.")


    dt_col = dt_candidates[0]
    log.info("DateTime column detected: '%s'", dt_col)
    df[dt_col] = pd.to_datetime(df[dt_col])  # infer_datetime_format removed in pandas 2.x
    df.rename(columns={dt_col: "datetime"}, inplace=True)

    # ── Detect inverter_id column ──────────────────────────────────────────
    # Prefer columns that explicitly name inverters or plants; avoid generic 'id'
    id_candidates = [c for c in df.columns
                     if any(k in c.lower() for k in ("inverter_id", "plant_id",
                                                      "device_id", "station_id",
                                                      "unit_id", "mac"))]
    if not id_candidates:
        # Broader fallback
        id_candidates = [c for c in df.columns
                         if any(k in c.lower() for k in ("inverter", "plant",
                                                          "device", "station",
                                                          "unit")) and "id" in c.lower()]
    if id_candidates:
        id_col = id_candidates[0]
        log.info("Inverter ID column detected: '%s'", id_col)
        df.rename(columns={id_col: "inverter_id"}, inplace=True)
    else:
        log.warning("No inverter_id column found – creating a dummy constant.")
        df["inverter_id"] = "INVERTER_1"

    # Sort chronologically within each inverter
    df.sort_values(["inverter_id", "datetime"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    log.info("Date range: %s → %s", df["datetime"].min(), df["datetime"].max())
    return df


# =============================================================================
# STEP 2 – CREATE FUTURE PREDICTION TARGET  (7-10 DAY WINDOW)
# =============================================================================

def _detect_failure_column(df: pd.DataFrame) -> str:
    """Heuristically find the column that encodes present-time failure events.

    Priority order:
      1. Numeric columns matching failure keywords with LOW positive rate
         (sparse events, < 50% non-zero) — best candidates for real failures
      2. Any binary (0/1) failure-keyword column regardless of rate
      3. Fall back to first keyword match

    NOTE: 'failure_label' is explicitly de-prioritised if it has > 50% ones,
    because it is a dense rolling metric, not a sparse failure event signal.
    """
    failure_keywords = ("failure", "fault", "alarm", "shutdown",
                        "error", "alert", "trip", "offline")
    candidates = [c for c in df.columns
                  if any(k in c.lower() for k in failure_keywords)
                  and c not in ("datetime", "inverter_id")
                  and pd.api.types.is_numeric_dtype(df[c])]

    if not candidates:
        raise ValueError(
            "Could not auto-detect a failure/alarm column. "
            "Please set FAILURE_COL manually in create_future_target()."
        )

    # Score each candidate: prefer sparse (low-positive-rate), numeric columns
    def _sparsity(col: str) -> float:
        """Fraction of non-zero values. Lower = sparser = better event signal."""
        nonzero: pd.Series = df[col].ne(0)
        return float(nonzero.mean())

    # Sort: sparse first, then prefer binary (nunique <= 2)
    scored = sorted(candidates, key=lambda c: (_sparsity(c), -(df[c].nunique() <= 2)))
    best = scored[0]
    rates = {c: f"{_sparsity(c)*100:.1f}%" for c in candidates}
    log.info("Failure column candidates with non-zero rate: %s  → using '%s'",
             rates, best)
    return best


def create_future_target(df: pd.DataFrame,
                         failure_col: str | None = None,
                         window_start_days: int = 7,
                         window_end_days:   int = 10) -> pd.DataFrame:
    """
    For each row at time t and inverter i, check whether ANY failure event
    occurs in the interval [t + 7d, t + 10d].

    Parameters
    ----------
    df               : DataFrame sorted by ['inverter_id', 'datetime']
    failure_col      : column encoding binary (0/1) or boolean failure events.
                       If None, auto-detected from column names.
    window_start_days: lower bound of the future window (default 7)
    window_end_days  : upper bound of the future window (default 10)

    Returns
    -------
    df with new column TARGET_COL = 'future_failure_7_10_days'
    """
    if failure_col is None:
        failure_col = _detect_failure_column(df)

    log.info("Building future target using column '%s' "
             "(window: +%dd → +%dd) …", failure_col, window_start_days, window_end_days)

    # Ensure failure column is numeric 0/1
    df[failure_col] = pd.to_numeric(df[failure_col], errors="coerce").fillna(0)
    df[TARGET_COL] = 0  # default: no upcoming failure

    # ── Vectorised window labelling (avoids O(n²) row loop) ───────────────
    # Strategy: for each inverter, build a list of failure timestamps, then
    # for every observation check whether any failure falls in (t+7d, t+10d].
    # We use searchsorted on the sorted failure timestamps for O(n log n).
    lo_delta = pd.Timedelta(days=window_start_days)
    hi_delta = pd.Timedelta(days=window_end_days)

    for inv_id, grp in df.groupby("inverter_id", sort=False):
        failure_times = grp.loc[
            grp[failure_col] > 0, "datetime"
        ].sort_values().values  # numpy datetime64 array of failure events

        if len(failure_times) == 0:
            # No failures for this inverter at all
            continue

        obs_times = grp["datetime"].values
        lo_times  = obs_times + lo_delta.to_timedelta64()
        hi_times  = obs_times + hi_delta.to_timedelta64()

        # searchsorted gives the index of first failure_time >= lo_time
        idx_lo = np.searchsorted(failure_times, lo_times,  side="left")
        idx_hi = np.searchsorted(failure_times, hi_times,  side="right")

        # A failure exists in the window if idx_lo < idx_hi
        labels = (idx_hi > idx_lo).astype(int)
        df.loc[grp.index, TARGET_COL] = labels

    pos = df[TARGET_COL].sum()
    log.info("Target distribution: positives=%d (%.2f%%)",
             pos, 100 * pos / len(df))
    return df


# =============================================================================
# STEP 3 – DATA CLEANING
# =============================================================================

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    • Drop duplicate rows
    • Drop columns that carry no predictive information
    • Handle missing values (median-fill numeric, mode-fill categorical)
    """
    log.info("Cleaning data …")

    # ── Drop duplicates ────────────────────────────────────────────────────
    before = len(df)
    df.drop_duplicates(inplace=True)
    log.info("  Dropped %d duplicate rows.", before - len(df))

    # ── Drop rows where datetime is null (index columns) ──────────────────
    df.dropna(subset=["datetime"], inplace=True)

    # ── Drop near-constant / near-zero-variance columns ───────────────────
    non_feature_cols = {"datetime", "inverter_id", TARGET_COL}
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    low_var = [c for c in numeric_cols
               if c not in non_feature_cols and df[c].nunique() <= 1]
    if low_var:
        log.info("  Dropping low-variance columns: %s", low_var)
        df.drop(columns=low_var, inplace=True)

    # ── Drop columns with > 60% missing ───────────────────────────────────
    missing_frac = df.isnull().mean()
    high_missing = missing_frac[missing_frac > 0.60].index.tolist()
    high_missing = [c for c in high_missing if c not in non_feature_cols]
    if high_missing:
        log.info("  Dropping high-missing columns (>60%%): %s", high_missing)
        df.drop(columns=high_missing, inplace=True)

    # ── Impute remaining numeric missing values with column median ─────────
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    medians = df[numeric_cols].median()
    df[numeric_cols] = df[numeric_cols].fillna(medians)

    # ── Impute categorical missing values with column mode ─────────────────
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in non_feature_cols]
    for c in cat_cols:
        mode_val = df[c].mode()
        if len(mode_val) > 0:
            df[c].fillna(mode_val[0], inplace=True)

    log.info("  Post-clean shape: %s", df.shape)
    return df


# =============================================================================
# STEP 4 – FEATURE ENGINEERING
# =============================================================================

def engineer_features(df: pd.DataFrame,
                       key_sensor_cols: list[str] | None = None) -> pd.DataFrame:
    """
    Compute time-series features per inverter:
      • rolling_mean_24h / rolling_std_24h
      • rolling_mean_7d  / rolling_std_7d
      • rate_of_change for key sensors
      • days_since_last_alarm
      • anomaly indicators  (z-score > 3)
    """
    log.info("Engineering features …")

    # ── Identify numeric sensor columns ───────────────────────────────────
    exclude = {"datetime", "inverter_id", TARGET_COL}
    all_numeric = [c for c in df.select_dtypes(include=[np.number]).columns
                   if c not in exclude]

    if key_sensor_cols is None:
        # Heuristic: treat all numeric columns as sensors
        key_sensor_cols = all_numeric

    df = df.sort_values(["inverter_id", "datetime"])

    def _rolling(grp, col, window, stat):
        """Generic rolling stat with a time-offset window."""
        return (grp[col]
                .rolling(window=window, min_periods=1)
                .agg(stat))

    new_feature_groups = []

    for inv_id, grp in df.groupby("inverter_id", sort=False):
        idx = grp.index

        # Set datetime as index temporarily for time-based rolling
        g = grp.set_index("datetime")

        feat = pd.DataFrame(index=idx)

        for col in key_sensor_cols:
            if col not in g.columns:
                continue
            series = g[col]

            # ── Rolling windows (time-based) ───────────────────────────
            feat[f"{col}_rolling_mean_24h"] = (
                series.rolling("24h", min_periods=1).mean().values)
            feat[f"{col}_rolling_std_24h"]  = (
                series.rolling("24h", min_periods=1).std().fillna(0).values)
            feat[f"{col}_rolling_mean_7d"]  = (
                series.rolling("7D",  min_periods=1).mean().values)
            feat[f"{col}_rolling_std_7d"]   = (
                series.rolling("7D",  min_periods=1).std().fillna(0).values)

            # ── Rate of change (first difference) ─────────────────────
            feat[f"{col}_roc"] = series.diff().fillna(0).values

            # ── Anomaly indicator  (|z-score| > 3) ────────────────────
            mu, sigma = series.mean(), series.std()
            if sigma > 0:
                z = (series - mu) / sigma
                feat[f"{col}_anomaly"] = (z.abs() > 3).astype(int).values
            else:
                feat[f"{col}_anomaly"] = 0

        # ── days_since_last_alarm ──────────────────────────────────────
        alarm_col = _detect_alarm_col(grp)
        if alarm_col:
            alarm_vals   = grp[alarm_col].fillna(0).values.astype(float)
            dt_vals      = grp["datetime"].values                 # datetime64
            n_obs        = len(grp)
            dsla_arr     = np.full(n_obs, np.nan)
            last_ns: int = -1   # nanoseconds since epoch of last alarm (-1 = none seen)

            for i in range(n_obs):
                if alarm_vals[i] > 0:
                    last_ns = int(dt_vals[i].astype("int64"))
                if last_ns >= 0:
                    cur_ns = int(dt_vals[i].astype("int64"))
                    dsla_arr[i] = (cur_ns - last_ns) / 8.64e13   # ns → days
            feat["days_since_last_alarm"] = dsla_arr
        else:
            feat["days_since_last_alarm"] = np.nan

        new_feature_groups.append(feat)

    # Concat all per-inverter feature frames and merge back
    all_new_feats = pd.concat(new_feature_groups)
    all_new_feats = all_new_feats.fillna(0)
    df = pd.concat([df, all_new_feats.reindex(df.index)], axis=1)

    # ── Remove duplicate column names (can happen if sensors overlap) ──────
    df = df.loc[:, ~df.columns.duplicated()]

    log.info("  Post-engineering shape: %s", df.shape)
    return df


def _detect_alarm_col(grp: pd.DataFrame) -> str | None:
    """Find an alarm/fault column in a group if present."""
    candidates = [c for c in grp.columns
                  if any(k in c.lower() for k in ("alarm", "fault",
                                                    "failure", "error",
                                                    "shutdown", "trip"))]
    return candidates[0] if candidates else None


# =============================================================================
# STEP 5 – FEATURE SELECTION
# =============================================================================

def select_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """
    X = all telemetry + engineered numeric features   (exclude datetime,
        inverter_id, and any remaining string cols)
    y = future_failure_7_10_days
    """
    log.info("Selecting features …")

    drop_always = {"datetime", "inverter_id", TARGET_COL}
    # Use select_dtypes to be explicit — this prevents MAC/string columns
    # from leaking into the feature matrix even if their dtype is 'object' or 'str'
    numeric_only = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_only if c not in drop_always]

    X = df[feature_cols].copy()
    y = df[TARGET_COL].copy()

    log.info("  Feature count: %d", len(feature_cols))
    log.info("  Class balance:\n%s", y.value_counts())
    return X, y, feature_cols


# =============================================================================
# STEP 6 – TIME-AWARE TRAIN / TEST SPLIT
# =============================================================================

def time_split(X: pd.DataFrame,
               y: pd.Series,
               train_ratio: float = 0.80) -> tuple:
    """
    Split WITHOUT shuffling.
    Earliest 80% → training set
    Latest  20% → test set
    """
    n       = len(X)
    cutoff  = int(n * train_ratio)

    X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
    y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]

    log.info("Train size: %d  |  Test size: %d", len(X_train), len(X_test))
    return X_train, X_test, y_train, y_test


In [9]:
# =============================================================================
# SOLAR INVERTER FAILURE PREDICTION PIPELINE
# =============================================================================

import os
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import shap
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

log = logging.getLogger("SolarML")

BASE_DIR = Path.cwd()

DATASET_PATH = BASE_DIR / "solar_ml_master_dataset.xlsx"
MODELS_DIR = BASE_DIR / "models"
PLOTS_DIR = BASE_DIR / "plots"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "future_failure_7_10_days"
RANDOM_STATE = 42
N_SPLITS_CV = 5

# =============================================================================
# LOAD DATASET
# =============================================================================

def load_dataset(path):

    log.info(f"Loading dataset from: {path}")

    if path.suffix == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    log.info(f"Dataset Shape: {df.shape}")
    return df


# =============================================================================
# PREPROCESS DATETIME
# =============================================================================

def preprocess_datetime(df):

    datetime_col = None

    for c in df.columns:
        if "time" in c.lower() or "date" in c.lower():
            datetime_col = c
            break

    if datetime_col is None:
        raise ValueError("No datetime column found")

    df[datetime_col] = pd.to_datetime(df[datetime_col])

    df.rename(columns={datetime_col: "datetime"}, inplace=True)

    if "inverter_id" not in df.columns:
        df["inverter_id"] = "INV_1"

    df = df.sort_values(["inverter_id", "datetime"])

    return df


# =============================================================================
# CREATE TARGET
# =============================================================================

def create_future_target(df):

    failure_col = None

    for c in df.columns:
        if "failure" in c.lower() or "fault" in c.lower():
            failure_col = c
            break

    if failure_col is None:
        raise ValueError("Failure column not found")

    df[failure_col] = df[failure_col].fillna(0)

    df[TARGET_COL] = df.groupby("inverter_id")[failure_col].shift(-48)

    df[TARGET_COL] = df[TARGET_COL].fillna(0)

    return df


# =============================================================================
# CLEAN DATA
# =============================================================================

def clean_data(df):

    df = df.drop_duplicates()

    numeric_cols = df.select_dtypes(include=np.number).columns

    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

    return df


# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

def engineer_features(df):

    sensor_cols = df.select_dtypes(include=np.number).columns

    sensor_cols = [c for c in sensor_cols if c not in [TARGET_COL]]

    for col in sensor_cols:

        df[f"{col}_rolling_mean"] = (
            df.groupby("inverter_id")[col]
            .rolling(24, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        df[f"{col}_diff"] = df.groupby("inverter_id")[col].diff().fillna(0)

    return df


# =============================================================================
# FEATURE SELECTION
# =============================================================================

def select_features(df):

    drop_cols = ["datetime", "inverter_id", TARGET_COL]

    numeric_cols = df.select_dtypes(include=np.number).columns

    feature_cols = [c for c in numeric_cols if c not in drop_cols]

    X = df[feature_cols]
    y = df[TARGET_COL]

    log.info(f"Total Features: {len(feature_cols)}")

    return X, y, feature_cols


# =============================================================================
# TIME SPLIT
# =============================================================================

def time_split(X, y):

    split = int(len(X) * 0.8)

    return X[:split], X[split:], y[:split], y[split:]


# =============================================================================
# BUILD MODELS
# =============================================================================

def build_models():

    models = {

        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            min_samples_leaf=10,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ),

        "XGBoost": XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.7,
            colsample_bytree=0.6,
            eval_metric="logloss",
            random_state=RANDOM_STATE
        ),

        "LightGBM": LGBMClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.05,
            num_leaves=31,
            random_state=RANDOM_STATE
        )
    }

    return models


# =============================================================================
# TRAIN MODELS
# =============================================================================

def train_all_models(models, X_train, y_train):

    fitted = {}

    for name, model in models.items():

        log.info(f"Training {name}")

        model.fit(X_train, y_train)

        fitted[name] = model

    return fitted


# =============================================================================
# EVALUATE
# =============================================================================

def evaluate_models(models, X_test, y_test):

    results = []

    for name, model in models.items():

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1]

        f1 = f1_score(y_test, y_pred)

        auc = roc_auc_score(y_test, y_prob)

        results.append([name, f1, auc])

        log.info(classification_report(y_test, y_pred))

    df = pd.DataFrame(results, columns=["Model","F1","AUC"])

    best_model = df.sort_values("F1", ascending=False).iloc[0]["Model"]

    return df, best_model


# =============================================================================
# SHAP
# =============================================================================

def explain_with_shap(model, X):

    sample = X.sample(min(2000, len(X)), random_state=42)

    explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(sample)

    shap.summary_plot(shap_values, sample, show=False)

    plt.savefig(PLOTS_DIR / "shap_summary.png")

    plt.close()


# =============================================================================
# SAVE MODEL
# =============================================================================

def save_artifacts(model, features):

    joblib.dump(model, MODELS_DIR / "trained_model.pkl")

    joblib.dump(features, MODELS_DIR / "feature_columns.pkl")

    log.info("Model saved")


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():

    log.info("Starting Training Pipeline")

    df = load_dataset(DATASET_PATH)

    df = preprocess_datetime(df)

    df = create_future_target(df)

    df = clean_data(df)

    df = engineer_features(df)

    X, y, features = select_features(df)

    X_train, X_test, y_train, y_test = time_split(X,y)

    models = build_models()

    fitted_models = train_all_models(models, X_train, y_train)

    metrics, best_name = evaluate_models(fitted_models, X_test, y_test)

    best_model = fitted_models[best_name]

    print(metrics)

    explain_with_shap(best_model, X_test)

    save_artifacts(best_model, features)

    log.info("Pipeline complete")


# =============================================================================

if __name__ == "__main__":
    main()

2026-03-06 10:28:05 [INFO] Starting Training Pipeline
2026-03-06 10:28:05 [INFO] Loading dataset from: c:\Users\Trupal Dholariya\OneDrive\Desktop\h2\solar_ml_master_dataset.xlsx
2026-03-06 10:38:46 [INFO] Dataset Shape: (1048575, 41)
2026-03-06 10:39:04 [INFO] Total Features: 96
2026-03-06 10:39:04 [INFO] Training RandomForest
2026-03-06 10:39:38 [INFO] Training XGBoost
2026-03-06 10:39:57 [INFO] Training LightGBM
  File "C:\Users\Trupal Dholariya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~

[LightGBM] [Info] Number of positive: 436772, number of negative: 402088
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.170023 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 20142
[LightGBM] [Info] Number of data points in the train set: 838860, number of used features: 93
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.520673 -> initscore=0.082740
[LightGBM] [Info] Start training from score 0.082740


2026-03-06 10:40:33 [INFO]               precision    recall  f1-score   support

         0.0       0.95      0.98      0.96     87313
         1.0       0.99      0.96      0.97    122402

    accuracy                           0.97    209715
   macro avg       0.97      0.97      0.97    209715
weighted avg       0.97      0.97      0.97    209715

2026-03-06 10:40:34 [INFO]               precision    recall  f1-score   support

         0.0       0.95      0.99      0.97     87313
         1.0       0.99      0.96      0.98    122402

    accuracy                           0.97    209715
   macro avg       0.97      0.97      0.97    209715
weighted avg       0.97      0.97      0.97    209715

2026-03-06 10:40:36 [INFO]               precision    recall  f1-score   support

         0.0       0.95      0.99      0.97     87313
         1.0       0.99      0.96      0.98    122402

    accuracy                           0.97    209715
   macro avg       0.97      0.97      0.97    

          Model        F1       AUC
0  RandomForest  0.973141  0.996108
1       XGBoost  0.975372  0.996469
2      LightGBM  0.976114  0.996438


2026-03-06 10:40:41 [INFO] Model saved
2026-03-06 10:40:41 [INFO] Pipeline complete
